In [1]:
# Import pandas for data manipulation and analysis (works with tabular data like CSVs)
import pandas as pd

# Import numpy for numerical operations (arrays, math functions, etc.)
import numpy as np

# Import itertools for advanced iteration tools (combinations, permutations, etc.)
import itertools

# Import text vectorizers from scikit-learn:
# - CountVectorizer: converts text into word counts
# - TfidfVectorizer: converts text into TF-IDF scores (importance of words)
# - HashingVectorizer: converts text into fixed-length feature vectors using hashing
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer

# Import train_test_split to split dataset into training and testing sets
from sklearn.model_selection import train_test_split

# Import PassiveAggressiveClassifier (a linear model useful for large-scale text classification)
from sklearn.linear_model import PassiveAggressiveClassifier

# Import Multinomial Naive Bayes (commonly used for text classification with word counts)
from sklearn.naive_bayes import MultinomialNB

# Import metrics for evaluating model performance (accuracy, confusion matrix, etc.)
from sklearn import metrics

# Import matplotlib for plotting graphs and visualizations
import matplotlib.pyplot as plt


In [2]:
# Read the CSV file named 'news.csv' into a pandas DataFrame and instruct pandas not to assign any column as index
df = pd.read_csv('news.csv', index_col=None)

In [3]:
print(df.head())

   Unnamed: 0                                              title  \
0        8476                       You Can Smell Hillary’s Fear   
1       10294  Watch The Exact Moment Paul Ryan Committed Pol...   
2        3608        Kerry to go to Paris in gesture of sympathy   
3       10142  Bernie supporters on Twitter erupt in anger ag...   
4         875   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  


In [4]:
dataset=df.drop("Unnamed: 0", axis=1)

In [5]:
print(dataset.head())

                                               title  \
0                       You Can Smell Hillary’s Fear   
1  Watch The Exact Moment Paul Ryan Committed Pol...   
2        Kerry to go to Paris in gesture of sympathy   
3  Bernie supporters on Twitter erupt in anger ag...   
4   The Battle of New York: Why This Primary Matters   

                                                text label  
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE  
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE  
2  U.S. Secretary of State John F. Kerry said Mon...  REAL  
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE  
4  It's primary day in New York and front-runners...  REAL  


In [6]:
y=dataset["label"]
X_train, X_test, y_train, y_test = train_test_split(dataset['text'], y, test_size=0.33, random_state=53)

In [8]:
# - Convert a collection of raw documents to a matrix of TF-IDF features
# - stop_words='english': removes common English words (like "the", "is", "and")
tfidf_Vectorizer = TfidfVectorizer(stop_words='english')
counter_Train = tfidf_Vectorizer.fit_transform(X_train)
print(counter_Train)
print(X_test)
count_test = tfidf_Vectorizer.transform(X_test)
print(count_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1119820 stored elements and shape (4244, 56922)>
  Coords	Values
  (1, 42470)	0.07711040274149526
  (1, 12105)	0.15008066461476866
  (1, 54177)	0.13782629144711137
  (1, 50628)	0.061296988343109586
  (1, 15924)	0.3479045460649079
  (1, 44520)	0.4973826512693341
  (1, 51896)	0.11596517664605868
  (1, 35783)	0.30902690818827977
  (1, 35256)	0.12628385718450857
  (1, 21881)	0.21271688045815978
  (1, 42534)	0.06081715886809217
  (1, 8399)	0.08729542880625335
  (1, 29531)	0.1454406205718245
  (1, 15927)	0.4973826512693341
  (1, 25686)	0.13550453594288983
  (1, 49203)	0.1672740861784377
  (1, 16814)	0.10404977746548139
  (1, 36087)	0.12648679854389897
  (1, 21568)	0.1007920919566398
  (1, 25684)	0.1030420922189754
  (1, 38823)	0.06048803110658644
  (1, 47506)	0.14539060877460044
  (1, 36831)	0.10772488937433067
  (2, 16972)	0.1606296088662543
  (2, 762)	0.48803966069171073
  :	:
  (4243, 41435)	0.02969665315895183
  (4243, 53607)	

In [9]:
# Get the number of unique features (words) learned by CountVectorizer
# - count_vectorizer.get_feature_names_out(): returns an array of all words in the vocabulary
#   → This equals the number of features (columns) in your training matrix
len(tfidf_Vectorizer.get_feature_names_out())

56922

In [10]:
# Step 1: Initialize the Multinomial Naive Bayes classifier
# - MultinomialNB is well-suited for text classification problems
#   where features are word counts or frequencies.
clf = MultinomialNB()

# Step 2: Train (fit) the classifier on the training data
# - count_train: the bag-of-words matrix for training documents
# - y_train: the true labels (e.g., FAKE or REAL) for training data
clf.fit(counter_Train, y_train)

# Step 3: Predict labels for the test data
# - count_test: the bag-of-words matrix for test documents
# - pred: the predicted labels for each test document
pred = clf.predict(count_test)

# Step 4: Calculate accuracy score
# - Compares predicted labels (pred) with true labels (y_test)
# - Returns the proportion of correct predictions
score = metrics.accuracy_score(y_test, pred)

# Step 5: Print accuracy with 3 decimal places
print("accuracy:   %0.3f" % score)

# Step 6: Generate confusion matrix
# - Compares true labels (y_test) with predicted labels (pred)
# - labels=['FAKE','REAL']: ensures matrix rows/columns are ordered
#   consistently for FAKE and REAL classes
cm = metrics.confusion_matrix(y_test, pred, labels=['FAKE', 'REAL'])

accuracy:   0.857


In [11]:
from sklearn.metrics import classification_report

report=classification_report(y_test, pred)

In [12]:
print(report)

              precision    recall  f1-score   support

        FAKE       0.96      0.73      0.83      1008
        REAL       0.80      0.97      0.88      1083

    accuracy                           0.86      2091
   macro avg       0.88      0.85      0.85      2091
weighted avg       0.88      0.86      0.85      2091

